# Build a NanoVLM from Scratch

## Chapter 33: Zero-Shot Classification

Read this chapter: [https://books.vizuara.ai/read/nanovlm-from-scratch/zero-shot-classification](https://books.vizuara.ai/read/nanovlm-from-scratch/zero-shot-classification)

---

**Generated file.** Extracted automatically from the book's own text. Do not hand-edit; edit the book instead.

Code cells appear in the order the blocks appear in the chapter, and may assume names defined in earlier cells or earlier chapters. Nothing here has been executed.

In [ ]:
# --- block 1 ---
import torch
import torch.nn.functional as F

@torch.no_grad()
def zero_shot_classify(model, image, labels, template="a photo of a {}"):
    # 1. Build one prompt per candidate label, encode to class vectors.
    prompts   = [template.format(label) for label in labels]
    text_emb  = model.encode_text(prompts)        # (num_labels, d), L2-normalized
    # 2. Encode the query image to a single vector.
    img_emb   = model.encode_image(image[None])   # (1, d), L2-normalized
    # 3. Cosine similarity == dot product (both are unit vectors).
    logit_scale = model.logit_scale.exp()          # learned temperature
    logits      = logit_scale * img_emb @ text_emb.T   # (1, num_labels)
    probs       = logits.softmax(dim=-1)           # class probabilities
    # 4. Argmax is the prediction.
    best = probs.argmax(dim=-1).item()
    return labels[best], probs[0]